In [1]:
from mylib.statistic_test import *

code_id = '0869 - Detailed Behavioral Analysis'
loc = join(figpath, 'Dsp', code_id)
mkdir(loc)

StatePalette = ["#89d858", "#8b66db", "#5796d5", "#eeb378"]
StatePalette2 = ['#b3d99b', '#d1c8e4', '#a3c1e0', '#ffcc99']
pass

d:\Softwares\Anaconda2025\envs\maze\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


        D:\Data\FinalResults\Dsp\0869 - Detailed Behavioral Analysis is already existed!


In [2]:
dir_template = join(loc, 'template')
mkdir(dir_template)
# Hidden Markov Model
import numpy as np
from hmmlearn.hmm import GaussianHMM

def get_hmm_states(
    behav_res_angles: np.ndarray, 
    behav_res_speeds: np.ndarray, 
    behav_speed_traj: np.ndarray,
    behav_lap_traj: np.ndarray
):
    """Use HMM to get behavioral states."""
    # Example data: 1D oscillatory behavior (you may have more features in real data)
    # Simulate some example data: 2 states with different distributions
    # Data: N time steps, observed as 1D feature (you can replace this with real data)

    behav_hidden_states = np.ones_like(behav_res_angles)
    
    observations = np.vstack([
        behav_res_angles,
        behav_res_speeds#,
        #behav_speed_traj
    ]).T

    fit_idx = np.where(
        (np.isnan(observations[:, 0]) == False) &
        (np.isnan(observations[:, 1]) == False)# &
        #(np.isnan(behav_speed_traj) == False)
    )[0]
    observations = observations[fit_idx, :]

    fit_lap_traj = behav_lap_traj[fit_idx]
    beg = np.concatenate([[0], np.where(np.diff(fit_lap_traj) != 0)[0] + 1])
    end = np.concatenate([np.where(np.diff(fit_lap_traj) != 0)[0]+1, [len(fit_lap_traj)]])
    lengths = end-beg

    # Define the number of hidden states (2 states for low and high amplitude)
    model = GaussianHMM(n_components=2, covariance_type="diag", n_iter=1000, random_state=42)

    # Train the model using the Baum-Welch algorithm (unsupervised)
    model.fit(observations, lengths=lengths)

    # Predict the hidden states for each time point
    hidden_states = model.predict(observations, lengths=lengths)
    
    idx0 = np.where(hidden_states == 0)[0]
    idx1 = np.where(hidden_states == 1)[0]
    
    res0 = np.nanmean(behav_res_angles[fit_idx][idx0])
    res1 = np.nanmean(behav_res_angles[fit_idx][idx1])
    if res0 > res1:
        behav_hidden_states[fit_idx] = 1 - hidden_states
    else:
        behav_hidden_states[fit_idx] = hidden_states
    return behav_hidden_states

def behavioral_template_retrieval(mouse: int, session: int, maze_type: int):
    with open(join(dir_template, f"{mouse}.pkl"), 'rb') as f:
        template = pickle.load(f)
        
    file_idx = np.where(f2['MiceID'] == mouse)[0][session]
    
    try:
        with open(f2['Trace File'][file_idx], 'rb') as f:
            trace = pickle.load(f)
    except:
        with open(f2['Trace Behav File'][file_idx], 'rb') as f:
            trace = pickle.load(f)
        
    beg, end = LapSplit(trace, trace['paradigm'])
    routes = classify_lap(spike_nodes_transform(trace['correct_nodes'], 12), beg, maze_type=maze_type)
    behav_nodes = spike_nodes_transform(trace['correct_nodes'], 12)
    
    behav_nodes_traj = []
    behav_nodes_son_traj = []
    behav_lap_traj = []
    behav_routes_traj = []
    behav_session_traj = []
    behav_params_traj = []
    behav_params_templ_traj = []
    behav_dt_traj = []
    behav_t_traj = []


    for j in range(len(beg)):
        dp = np.diff(trace['correct_pos'][beg[j]:end[j], :], axis=0)/10
        dt = np.diff(trace['correct_time'][beg[j]:end[j]])/1000
        dl = np.sqrt(dp[:, 0]**2 + dp[:, 1]**2)
        v = dl/dt
        # Moving Direction
        a = np.arctan2(dp[:, 1], dp[:, 0])
        
        params = np.vstack([
            trace['correct_pos'][beg[j]:end[j]-1, :].T/10,
            a,
            v
        ])
        params_templ = template[behav_nodes[beg[j]:end[j]-1]-1, :, 0].T
        
        behav_nodes_traj.append(behav_nodes[beg[j]:end[j]-1])
        behav_nodes_son_traj.append(trace['correct_nodes'][beg[j]:end[j]-1])
        behav_lap_traj.append(np.repeat(j, len(params[0])))
        behav_routes_traj.append(np.repeat(routes[j], len(params[0])))
        behav_session_traj.append(np.repeat(session, len(params[0])))
        behav_params_traj.append(params)
        behav_params_templ_traj.append(params_templ)
        behav_dt_traj.append(dt)
        behav_t_traj.append((trace['correct_time'][beg[j]:end[j]-1] - trace['correct_time'][beg[j]])/1000)
        
    behav_nodes_traj = np.concatenate(behav_nodes_traj)
    incorrect_path_idx = np.where(np.isin(behav_nodes_traj, CP_DSPs[maze_type][0]) == False)[0]
    behav_nodes_son_traj = np.concatenate(behav_nodes_son_traj)
    behav_lap_traj = np.concatenate(behav_lap_traj)
    behav_routes_traj = np.concatenate(behav_routes_traj)
    behav_session_traj = np.concatenate(behav_session_traj)
    behav_params_traj = np.concatenate(behav_params_traj, axis=1)
    behav_params_templ_traj = np.concatenate(behav_params_templ_traj, axis=1)
    behav_dt_traj = np.concatenate(behav_dt_traj)
    behav_t_traj = np.concatenate(behav_t_traj)

    behav_res_angles = behav_params_traj[2, :] - behav_params_templ_traj[2, :]
    behav_res_angles[behav_res_angles > np.pi] = 2*np.pi - behav_res_angles[behav_res_angles > np.pi]
    behav_res_angles[behav_res_angles < -np.pi] = -2*np.pi - behav_res_angles[behav_res_angles < -np.pi]
    behav_res_angles = np.rad2deg(np.abs(behav_res_angles))
    behav_res_speeds = behav_params_traj[3, :] - behav_params_templ_traj[3, :]

    def gaussian_kernel(x, sigma):
        return np.exp(-x**2/(2*sigma**2))/(sigma*np.sqrt(2*np.pi))

    # Smooth the residuals within each lap

    beg = np.concatenate([[0], np.where(np.diff(behav_lap_traj) != 0)[0] + 1])
    end = np.concatenate([np.where(np.diff(behav_lap_traj) != 0)[0]+1, [len(behav_lap_traj)]])
    for i in range(len(beg)):
        gkernel = gaussian_kernel(np.linspace(-5, 5, 11), sigma=1)
        gkernel /= np.sum(gkernel)
        nonnan_idx = np.where(np.isnan(behav_res_angles[beg[i]:end[i]]) == False)[0]
        try:
            behav_res_angles[beg[i]:end[i]][nonnan_idx] = np.convolve(behav_res_angles[beg[i]:end[i]][nonnan_idx], gkernel, mode='same')
            behav_res_speeds[beg[i]:end[i]][nonnan_idx] = np.convolve(behav_res_speeds[beg[i]:end[i]][nonnan_idx], gkernel, mode='same')
        except:
            pass

    # Set the incorrect path indices to NaN
    #behav_res_angles[incorrect_path_idx] = np.nan
    #behav_res_speeds[incorrect_path_idx] = np.nan
    
    behav_hidden_states = get_hmm_states(
        behav_res_angles,   
        behav_res_speeds,
        behav_params_traj[3, :],
        behav_lap_traj
    )
    #behav_hidden_states[incorrect_path_idx] = 1
    
    return (
        behav_nodes_traj.astype(np.int64),
        behav_nodes_son_traj.astype(np.int64),
        behav_lap_traj.astype(np.int64),
        behav_routes_traj.astype(np.int64),
        behav_session_traj.astype(np.int64),
        behav_params_traj.astype(np.float64),
        behav_params_templ_traj.astype(np.float64),
        behav_dt_traj.astype(np.float64),
        behav_t_traj.astype(np.float64),
        behav_res_angles.astype(np.float64),
        behav_res_speeds.astype(np.float64),
        behav_hidden_states.astype(np.float64)
   )
    
    
(
    behav_nodes_traj,
    behav_nodes_son_traj,
    behav_lap_traj,
    behav_routes_traj,
    behav_session_traj,
    behav_params_traj,
    behav_params_templ_traj,
    behav_dt_traj,
    behav_t_traj,
    behav_res_angles,
    behav_res_speeds,
    behav_hidden_states
) = behavioral_template_retrieval(10266, session=0, maze_type=2)

fig = plt.figure(figsize=(4, 2))
ax = Clear_Axes(plt.axes(), close_spines=['top', 'right'], ifxticks=True, ifyticks=True)
maze_type = 2
for r in tqdm(range(7)):
    prob = np.zeros((2, CP_DSPs[maze_type][r].shape[0]-1), np.float64)
    for i, b in enumerate(CP_DSPs[maze_type][r][:-1]):
        retrieved_state_idx = np.where((behav_routes_traj == r) & (behav_nodes_traj == b) & (behav_hidden_states == 0))[0]
        other_state_idx = np.where((behav_routes_traj == r) & (behav_nodes_traj == b) & (behav_hidden_states == 1))[0]
        
        prob[0, i] = np.nansum(behav_dt_traj[retrieved_state_idx])
        prob[1, i] = np.nansum(behav_dt_traj[other_state_idx])
            
    prob[0, :] = np.convolve(prob[0, :], np.ones(7), mode='same')
    prob[1, :] = np.convolve(prob[1, :], np.ones(7), mode='same')

    prob = prob / (np.sum(prob, axis=0) + 1e-8)
    _len = prob.shape[1]
    ax.plot(np.arange(100)[-_len:], prob[0, :], color=DSPPalette[r], linewidth=0.5)
ax.set_ylim(0, 1)
ax.set_yticks(np.linspace(0, 1, 6))
ax.set_title("Maze 2")
ax.set_ylabel("Behavioral Retrieval")
ax.set_xlabel("Position Along Track (Bins)")
plt.savefig( join(loc, f"HMM State Probability Along Track [Maze 2].png"), dpi=2400)
plt.savefig( join(loc, f"HMM State Probability Along Track [Maze 2].svg"), dpi=600)
plt.show()

        D:\Data\FinalResults\Dsp\0869 - Detailed Behavioral Analysis\template is already existed!


FileNotFoundError: [Errno 2] No such file or directory: 'D:\\Data\\FinalResults\\Dsp\\0869 - Detailed Behavioral Analysis\\template\\10266.pkl'